# Minimum Spanning Tree Forecasting Portfolio Comparison

This research notebook compares stock-selection portfolios using a minimum spanning tree. It estimates links from historical return series and compares simple momentum, ARIMA and neural autoregression with VaR or Sharpe-based weighting. Each backtest uses the original fixed stock basket and adjusted Yahoo Finance history. The return report contains historical benchmark calculations and may reflect survivorship in the fixed basket.


In [ ]:
import numpy as np
import pandas as pd
from _market_data import download_ohlcv
import warnings
from typing import Dict, List, Tuple, Optional, Union
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from statsmodels.tsa.api import VAR
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import time
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# =============================================================================
# CONFIGURATION & DATA STRUCTURES
# =============================================================================

@dataclass
class PortfolioConfig:
    """Configuration class for portfolio parameters"""
    window_size: int = 60  # Reduced from 120 for faster processing
    forecast_horizon: int = 5  # Reduced from 10 for speed
    top_k_stocks: int = 5  # Number of stocks to select
    var_confidence: float = 0.05  # VaR confidence level
    initial_capital: float = 100000.0
    max_workers: int = 4  # Reduced for rate limiting
    seed_range: Tuple[int, int] = (99, 140)  # Random seed range for NNAR

@dataclass
class MarketIndex:
    """Market index information"""
    symbol: str
    name: str
    country: str
    constituents_endpoint: Optional[str] = None

# Major world indices configuration
WORLD_INDICES = [
    MarketIndex("^GSPC", "S&P 500", "USA"),
    MarketIndex("^IXIC", "NASDAQ Composite", "USA"),
    MarketIndex("^DJI", "Dow Jones Industrial", "USA"),
    MarketIndex("^FTSE", "FTSE 100", "UK"),
    MarketIndex("^GDAXI", "DAX", "Germany"),
    MarketIndex("^FCHI", "CAC 40", "France"),
    MarketIndex("^N225", "Nikkei 225", "Japan"),
    MarketIndex("^HSI", "Hang Seng", "Hong Kong"),
    MarketIndex("^STOXX50E", "EURO STOXX 50", "Europe"),
]

# =============================================================================
# DATA FETCHER
# =============================================================================

class YahooDataFetcher:
    """Downloads historical adjusted prices through yfinance."""

    def __init__(self, config: PortfolioConfig):
        self.config = config


    def get_available_stocks(self) -> List[str]:
        """Return the original fixed research universe."""
        try:
            # Use S&P 500 constituents to reduce API calls
            sp500_symbols = [
                'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'BRK-B', 'UNH', 'JNJ',
                'JPM', 'V', 'PG', 'XOM', 'HD', 'CVX', 'MA', 'BAC', 'ABBV', 'PFE',
                'KO', 'AVGO', 'PEP', 'TMO', 'COST', 'MRK', 'WMT', 'CSCO', 'ACN', 'DHR',
                'VZ', 'ADBE', 'NEE', 'CRM', 'ABT', 'NKE', 'TXN', 'RTX', 'QCOM', 'CMCSA',
                'AMGN', 'HON', 'SPGI', 'UNP', 'PM', 'GE', 'IBM', 'CAT', 'GILD', 'BA',
                'AMD', 'ISRG', 'INTC', 'SYK', 'C', 'NOW', 'BKNG', 'LOW', 'GS', 'AMAT',
                'TJX', 'BLK', 'AXP', 'SBUX', 'REGN', 'CVS', 'PYPL', 'MDLZ', 'PLD', 'CB',
                'DE', 'MMC', 'AMT', 'ADI', 'INTU', 'SO', 'MO', 'ZTS', 'SCHW', 'CI',
                'CL', 'FIS', 'CSX', 'BSX', 'DUK', 'NOC', 'MU', 'EQIX', 'ICE', 'WM',
                'COP', 'NSC', 'AON', 'USB', 'D', 'GD', 'ITW', 'FISV', 'SHW', 'EMR',
                'F', 'FCX', 'PSA', 'GM', 'ECL', 'HCA', 'TGT', 'CCI', 'ATVI', 'EOG'
            ]
            logger.info(f"Using curated list of {len(sp500_symbols)} S&P 500 symbols")
            return sp500_symbols
        except Exception as e:
            logger.error(f"Failed to get stock symbols: {e}")
            # Fallback to minimal set
            return ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'BRK-B', 'UNH', 'JPM']

    def get_historical_data(self, symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
        try:
            prices = download_ohlcv(symbol, start_date, end_date, auto_adjust=True)
        except RuntimeError as error:
            logger.warning("Unable to download %s: %s", symbol, error)
            return pd.DataFrame()
        frame = prices["Close"].rename("adjClose").rename_axis("date").reset_index()
        frame["return"] = frame["adjClose"].pct_change()
        return frame.dropna(subset=["return"])

    def get_batch_historical_data(self, symbols: List[str], start_date: str, end_date: str) -> Dict[str, pd.DataFrame]:
        results = {}
        for symbol in symbols:
            history = self.get_historical_data(symbol, start_date, end_date)
            if len(history) > self.config.window_size:
                results[symbol] = history
            else:
                logger.warning("Insufficient historical prices for %s.", symbol)
        return results

# =============================================================================
# VAR & FEVD IMPLEMENTATION
# =============================================================================

class VARFEVDAnalyzer:
    """Vector Autoregression and Forecast Error Variance Decomposition"""

    def __init__(self, forecast_horizon: int = 10):
        self.forecast_horizon = forecast_horizon

    def fit_bivariate_var(self, data1: pd.Series, data2: pd.Series, lag_order: int = 1) -> Optional[VAR]:
        """Fit bivariate VAR model - robust version with detailed debugging"""
        try:
            # Combine series and clean data
            df = pd.concat([data1.rename('stock1'), data2.rename('stock2')], axis=1).dropna()

            if len(df) < 25:  # Minimum observations
                return None

            # Check for constant series (no variation)
            if df.std().min() < 1e-8:
                return None

            # Check for extreme values
            if (df.abs() > 10).any().any():  # More than 1000% daily return
                return None

            # Use robust subset of data
            if len(df) > 50:
                df = df.tail(50)  # Use last 50 observations max

            try:
                # Fit VAR model with minimal requirements
                model = VAR(df)
                # Try lag 1 first, then lag 0 if it fails
                try:
                    fitted_model = model.fit(maxlags=1, ic=None)
                except:
                    # Fallback to VAR(0) - essentially no lag dependencies
                    fitted_model = model.fit(maxlags=0, ic=None)

                return fitted_model

            except Exception as e:
                return None

        except Exception as e:
            return None

    def compute_fevd(self, var_model: VAR, horizon: int) -> np.ndarray:
        """Compute FEVD - robust version with fallbacks"""
        try:
            # Use very short horizon for stability
            effective_horizon = min(horizon, 2)
            fevd_result = var_model.fevd(effective_horizon)
            return fevd_result.decomp
        except:
            try:
                # Fallback: try horizon 1
                fevd_result = var_model.fevd(1)
                return fevd_result.decomp
            except:
                # Final fallback: return simple correlation-based influence
                try:
                    # Extract residuals for correlation analysis
                    resid = var_model.resid
                    if len(resid) > 5:
                        corr = np.corrcoef(resid.T)
                        # Convert correlation to FEVD-like format
                        fevd_fake = np.zeros((1, 2, 2))
                        fevd_fake[0, 0, 0] = 0.7  # Self influence
                        fevd_fake[0, 1, 1] = 0.7  # Self influence
                        fevd_fake[0, 0, 1] = min(0.3, abs(corr[0, 1]))  # Cross influence
                        fevd_fake[0, 1, 0] = min(0.3, abs(corr[1, 0]))  # Cross influence
                        return fevd_fake
                except:
                    pass
                return np.zeros((1, 2, 2))

    def build_influence_matrix(self, return_data: Dict[str, pd.Series], window_start: int, window_end: int) -> pd.DataFrame:
        """Build influence matrix using FEVD - robust version with detailed diagnostics"""
        symbols = list(return_data.keys())
        n_symbols = len(symbols)
        influence_matrix = pd.DataFrame(np.eye(n_symbols), index=symbols, columns=symbols)

        # Calculate total pairs for progress tracking
        total_pairs = (n_symbols * (n_symbols - 1)) // 2
        logger.info(f"Building influence matrix: {n_symbols} symbols, {total_pairs} pairs to process")

        processed_pairs = 0
        successful_pairs = 0
        var_fit_failures = 0
        fevd_failures = 0
        data_insufficient = 0

        for i, sym1 in enumerate(symbols):
            for j, sym2 in enumerate(symbols[i+1:], i+1):  # Only upper triangle
                processed_pairs += 1

                try:
                    # Get window data and validate
                    data1 = return_data[sym1].iloc[window_start:window_end]
                    data2 = return_data[sym2].iloc[window_start:window_end]

                    # Data validation
                    if len(data1) < 25 or len(data2) < 25:
                        data_insufficient += 1
                        continue

                    # Additional data quality checks
                    if data1.isna().sum() > len(data1) * 0.1 or data2.isna().sum() > len(data2) * 0.1:
                        data_insufficient += 1
                        continue

                    # Check for sufficient variation
                    if data1.std() < 1e-6 or data2.std() < 1e-6:
                        data_insufficient += 1
                        continue

                    # Fit VAR model with debugging
                    var_model = self.fit_bivariate_var(data1, data2)
                    if var_model is None:
                        var_fit_failures += 1
                        continue

                    # Compute FEVD with debugging
                    fevd = self.compute_fevd(var_model, self.forecast_horizon)
                    if fevd is None or fevd.shape[0] == 0:
                        fevd_failures += 1
                        continue

                    # Extract influences - use last available period
                    if fevd.shape[0] > 0:
                        last_period = -1
                        # Bidirectional influences
                        influence_12 = fevd[last_period, 1, 0] if fevd.shape[1] > 1 else 0.0
                        influence_21 = fevd[last_period, 0, 1] if fevd.shape[1] > 1 else 0.0

                        # Validate influences are reasonable
                        influence_12 = max(0.0, min(0.5, influence_12))  # Cap at 50%
                        influence_21 = max(0.0, min(0.5, influence_21))

                        if influence_12 > 1e-6 or influence_21 > 1e-6:  # Non-trivial influences
                            influence_matrix.iloc[j, i] = influence_12  # sym1 -> sym2
                            influence_matrix.iloc[i, j] = influence_21  # sym2 -> sym1
                            successful_pairs += 1

                except Exception as e:
                    # Log first few errors for debugging
                    if processed_pairs <= 5:
                        logger.warning(f"Error processing pair {sym1}-{sym2}: {str(e)[:100]}")
                    continue

                # Progress update every 20 pairs
                if processed_pairs % 20 == 0:
                    progress = (processed_pairs / total_pairs) * 100
                    logger.info(f"FEVD Progress: {processed_pairs}/{total_pairs} ({progress:.1f}%), {successful_pairs} successful")

        # Final diagnostics
        success_rate = (successful_pairs / total_pairs) * 100 if total_pairs > 0 else 0
        logger.info(f"Influence matrix complete:")
        logger.info(f"  Total pairs: {total_pairs}")
        logger.info(f"  Successful: {successful_pairs} ({success_rate:.1f}%)")
        logger.info(f"  Failures - Insufficient data: {data_insufficient}")
        logger.info(f"  Failures - VAR fitting: {var_fit_failures}")
        logger.info(f"  Failures - FEVD computation: {fevd_failures}")

        # If very low success rate, try fallback correlation method
        if successful_pairs < total_pairs * 0.1:  # Less than 10% success
            logger.warning(f"Low FEVD success rate ({success_rate:.1f}%), using correlation fallback")
            influence_matrix = self._build_correlation_influence_matrix(return_data, window_start, window_end)

        return influence_matrix

    def _build_correlation_influence_matrix(self, return_data: Dict[str, pd.Series], window_start: int, window_end: int) -> pd.DataFrame:
        """Fallback method using correlation-based influences"""
        symbols = list(return_data.keys())
        influence_matrix = pd.DataFrame(0.0, index=symbols, columns=symbols)

        # Build correlation matrix
        returns_df = pd.concat({sym: return_data[sym].iloc[window_start:window_end] for sym in symbols}, axis=1)
        correlation_matrix = returns_df.corr().fillna(0)

        # Convert correlations to influences (absolute correlation as proxy)
        for i, sym1 in enumerate(symbols):
            for j, sym2 in enumerate(symbols):
                if i == j:
                    influence_matrix.iloc[i, j] = 1.0  # Self-influence
                else:
                    # Use absolute correlation as influence strength
                    correlation = abs(correlation_matrix.iloc[i, j])
                    influence = min(0.3, correlation * 0.5)  # Scale and cap at 30%
                    influence_matrix.iloc[i, j] = influence

        logger.info(f"Correlation-based influence matrix: {(influence_matrix > 0.01).sum().sum() - len(symbols)} non-trivial influences")
        return influence_matrix

# =============================================================================
# MST NETWORK CONSTRUCTION
# =============================================================================

class MSTNetworkBuilder:
    """Minimum Spanning Tree network construction"""

    @staticmethod
    def influence_to_cost_matrix(influence_matrix: pd.DataFrame) -> pd.DataFrame:
        """Convert influence matrix to cost matrix"""
        # Cost = 1 - influence (higher influence = lower cost)
        cost_matrix = 1.0 - influence_matrix.clip(0, 1)

        # Symmetrize by taking minimum cost
        symbols = cost_matrix.index
        for i in symbols:
            for j in symbols:
                if i != j:
                    min_cost = min(cost_matrix.loc[i, j], cost_matrix.loc[j, i])
                    cost_matrix.loc[i, j] = min_cost
                    cost_matrix.loc[j, i] = min_cost

        return cost_matrix

    @staticmethod
    def build_mst(cost_matrix: pd.DataFrame) -> nx.Graph:
        """Build Minimum Spanning Tree using Prim's algorithm"""
        symbols = list(cost_matrix.index)
        n = len(symbols)

        if n < 2:
            return nx.Graph()

        # Create complete graph with costs as weights
        G = nx.Graph()
        for i in range(n):
            for j in range(i+1, n):
                weight = cost_matrix.iloc[i, j]
                if not np.isnan(weight) and weight < np.inf:
                    G.add_edge(symbols[i], symbols[j], weight=weight)

        # Compute MST
        if len(G.edges()) > 0:
            mst = nx.minimum_spanning_tree(G, weight='weight')
            return mst
        else:
            return nx.Graph()

    @staticmethod
    def calculate_centrality(mst: nx.Graph) -> Dict[str, float]:
        """Calculate degree centrality for MST nodes"""
        if len(mst.nodes()) == 0:
            return {}
        return nx.degree_centrality(mst)

# =============================================================================
# FORECASTING MODELS
# =============================================================================

class ForecastingModels:
    """Time series forecasting models: ARIMA and NNAR"""

    def __init__(self, random_seed: int = 42):
        self.random_seed = random_seed
        np.random.seed(random_seed)

    def fit_arima(self, data: pd.Series, max_order: Tuple[int, int, int] = (1, 1, 1)) -> Optional[ARIMA]:
        """Fit ARIMA model - simplified for speed"""
        try:
            # Use simple ARIMA(1,1,1) for speed
            model = ARIMA(data, order=(1, 1, 1))
            fitted = model.fit()
            return fitted
        except:
            try:
                # Fallback to even simpler model
                model = ARIMA(data, order=(1, 0, 0))
                fitted = model.fit()
                return fitted
            except:
                return None

    def fit_nnar(self, data: pd.Series, lags: int = 3, hidden_neurons: int = 3) -> Optional[MLPRegressor]:
        """Fit NNAR model - simplified for speed"""
        try:
            # Prepare lagged features with fewer lags
            X, y = [], []
            for i in range(lags, len(data)):
                X.append(data.iloc[i-lags:i].values)
                y.append(data.iloc[i])

            if len(X) < 15:  # Reduced minimum samples
                return None

            X, y = np.array(X), np.array(y)

            # Fit simpler neural network
            model = MLPRegressor(
                hidden_layer_sizes=(hidden_neurons,),
                activation='tanh',
                max_iter=50,  # Reduced from 200
                random_state=self.random_seed,
                alpha=0.1,    # Higher regularization
                learning_rate_init=0.01,
                early_stopping=True,
                n_iter_no_change=5
            )
            model.fit(X, y)
            return model

        except:
            return None

    def forecast_arima(self, model: ARIMA, steps: int = 1) -> float:
        """Generate ARIMA forecast"""
        try:
            forecast = model.forecast(steps=steps)
            return float(np.asarray(forecast)[0]) if len(forecast) > 0 else 0.0
        except:
            return 0.0

    def forecast_nnar(self, model: MLPRegressor, data: pd.Series) -> float:
        """Generate NNAR forecast"""
        try:
            lags = model.n_features_in_
            if len(data) < lags:
                return 0.0
            last_values = data.iloc[-lags:].values.reshape(1, -1)
            forecast = model.predict(last_values)
            return float(forecast[0])
        except:
            return 0.0

# =============================================================================
# RISK MANAGEMENT
# =============================================================================

class RiskManager:
    """Risk management utilities"""

    @staticmethod
    def calculate_historical_var(returns: pd.Series, confidence: float = 0.05) -> float:
        """Calculate historical Value at Risk"""
        if len(returns) == 0:
            return 10.0  # High penalty for insufficient data

        returns_clean = returns.dropna()
        if len(returns_clean) == 0:
            return 10.0

        var = -np.percentile(returns_clean, confidence * 100)
        return max(0.001, var)  # Minimum positive value

    @staticmethod
    def calculate_sharpe_ratio(returns: pd.Series, risk_free_rate: float = 0.0) -> float:
        """Calculate Sharpe ratio"""
        if len(returns) == 0:
            return 0.0

        returns_clean = returns.dropna()
        if len(returns_clean) == 0:
            return 0.0

        mean_return = returns_clean.mean()
        std_return = returns_clean.std()

        if std_return == 0:
            return 0.0

        return (mean_return - risk_free_rate) / std_return

    @staticmethod
    def calculate_portfolio_weights(selected_stocks: List[str], return_data: Dict[str, pd.Series],
                                  window_start: int, window_end: int, method: str = 'var') -> Dict[str, float]:
        """Calculate portfolio weights using VaR or Sharpe ratio"""
        weights = {}

        if method == 'var':
            # Inverse VaR weighting
            vars = {}
            for stock in selected_stocks:
                stock_returns = return_data[stock].iloc[window_start:window_end]
                vars[stock] = RiskManager.calculate_historical_var(stock_returns)

            # Inverse weighting (lower risk = higher weight)
            inv_vars = {stock: 1.0 / var_val for stock, var_val in vars.items()}
            total_inv_var = sum(inv_vars.values())

            if total_inv_var > 0:
                weights = {stock: inv_var / total_inv_var for stock, inv_var in inv_vars.items()}
            else:
                # Equal weights fallback
                weights = {stock: 1.0 / len(selected_stocks) for stock in selected_stocks}

        elif method == 'sharpe':
            # Sharpe ratio weighting
            sharpes = {}
            for stock in selected_stocks:
                stock_returns = return_data[stock].iloc[window_start:window_end]
                sharpes[stock] = max(0.0, RiskManager.calculate_sharpe_ratio(stock_returns))

            total_sharpe = sum(sharpes.values())
            if total_sharpe > 0:
                weights = {stock: sharpe / total_sharpe for stock, sharpe in sharpes.items()}
            else:
                # Equal weights fallback
                weights = {stock: 1.0 / len(selected_stocks) for stock in selected_stocks}

        return weights

# =============================================================================
# PORTFOLIO STRATEGIES
# =============================================================================

class PortfolioStrategy:
    """Portfolio strategy implementation"""

    def __init__(self, config: PortfolioConfig):
        self.config = config
        self.data_fetcher = YahooDataFetcher(config)
        self.var_fevd = VARFEVDAnalyzer(config.forecast_horizon)
        self.forecasting = ForecastingModels()

    def run_strategy(self, strategy_name: str, return_data: Dict[str, pd.Series],
                    price_data: Dict[str, pd.Series]) -> Dict[str, any]:
        """Run specific portfolio strategy with detailed debugging"""

        symbols = list(return_data.keys())
        n_periods = len(next(iter(return_data.values())))

        logger.info(f"🎯 Starting strategy: {strategy_name}")
        logger.info(f"📊 Data shape: {len(symbols)} symbols × {n_periods} periods")
        logger.info(f"📅 Trading window: {self.config.window_size} to {n_periods-1}")

        # Initialize tracking variables
        portfolio_values = []
        selected_stocks_history = []
        weights_history = []
        debug_info = []  # For debugging

        initial_capital = self.config.initial_capital
        current_capital = initial_capital
        successful_trades = 0
        total_periods = 0

        # Strategy execution
        for t in range(self.config.window_size, n_periods - 1):
            total_periods += 1
            window_start = t - self.config.window_size
            window_end = t

            period_debug = {
                'period': t,
                'window': f"{window_start}:{window_end}",
                'status': 'started'
            }

            try:
                # Step 1: Build MST network
                logger.info(f"📈 Period {t}: Building influence matrix...")
                influence_matrix = self.var_fevd.build_influence_matrix(
                    return_data, window_start, window_end
                )

                # Debug influence matrix
                non_zero_influences = (influence_matrix != 0).sum().sum() - len(symbols)  # Exclude diagonal
                period_debug['influences'] = int(non_zero_influences)
                logger.info(f"   💡 Non-zero influences: {non_zero_influences}/{len(symbols)**2 - len(symbols)}")

                if non_zero_influences == 0:
                    logger.warning(f"   ⚠️  No influences detected, skipping period {t}")
                    period_debug['status'] = 'no_influences'
                    debug_info.append(period_debug)
                    portfolio_values.append(current_capital)
                    selected_stocks_history.append([])
                    weights_history.append({})
                    continue

                cost_matrix = MSTNetworkBuilder.influence_to_cost_matrix(influence_matrix)
                mst = MSTNetworkBuilder.build_mst(cost_matrix)
                centrality = MSTNetworkBuilder.calculate_centrality(mst)

                # Debug MST
                period_debug['mst_edges'] = len(mst.edges())
                period_debug['centrality_nodes'] = len(centrality)
                logger.info(f"   🌲 MST edges: {len(mst.edges())}, Centrality nodes: {len(centrality)}")

                # Step 2: Select top-k central stocks
                if not centrality:
                    selected_stocks = symbols[:self.config.top_k_stocks]
                    logger.warning(f"   ⚠️  No centrality, using first {len(selected_stocks)} stocks")
                else:
                    sorted_stocks = sorted(centrality.items(), key=lambda x: x[1], reverse=True)
                    selected_stocks = [stock for stock, _ in sorted_stocks[:self.config.top_k_stocks]]
                    logger.info(f"   🎯 Selected stocks: {selected_stocks}")
                    top_centralities = [centrality[s] for s in selected_stocks[:3]]
                    logger.info(f"   📊 Top centralities: {[f'{c:.3f}' for c in top_centralities]}")

                period_debug['selected_stocks'] = selected_stocks

                # Step 3: Generate forecasts and trading signals
                forecasts = {}
                trading_signals = {}

                for stock in selected_stocks:
                    stock_returns = return_data[stock].iloc[window_start:window_end]

                    if 'arima' in strategy_name.lower():
                        model = self.forecasting.fit_arima(stock_returns)
                        forecast = self.forecasting.forecast_arima(model) if model else 0.0
                    elif 'nnar' in strategy_name.lower():
                        model = self.forecasting.fit_nnar(stock_returns)
                        forecast = self.forecasting.forecast_nnar(model, stock_returns) if model else 0.0
                    else:
                        # For basic MST strategies, use simple momentum forecast
                        recent_returns = stock_returns.tail(10)  # Last 10 days
                        if len(recent_returns) > 0:
                            forecast = recent_returns.mean()  # Simple momentum
                        else:
                            forecast = 0.0

                    forecasts[stock] = forecast
                    trading_signals[stock] = 1 if forecast > 0 else (-1 if forecast < 0 else 0)

                # Debug forecasts
                positive_forecasts = sum(1 for f in forecasts.values() if f > 0)
                negative_forecasts = sum(1 for f in forecasts.values() if f < 0)
                period_debug['forecasts'] = {
                    'positive': positive_forecasts,
                    'negative': negative_forecasts,
                    'values': {k: f"{v:.6f}" for k, v in list(forecasts.items())[:3]}
                }
                logger.info(f"   🔮 Forecasts: {positive_forecasts}+ / {negative_forecasts}- / {len(forecasts)-positive_forecasts-negative_forecasts}=0")

                # Step 4: Calculate weights
                weight_method = 'sharpe' if 'sharpe' in strategy_name.lower() else 'var'
                raw_weights = RiskManager.calculate_portfolio_weights(
                    selected_stocks, return_data, window_start, window_end, weight_method
                )

                # Debug raw weights
                logger.info(f"   ⚖️  Raw weights ({weight_method}): {[(k, f'{v:.3f}') for k, v in list(raw_weights.items())[:3]]}")
                period_debug['raw_weights_sum'] = sum(raw_weights.values())

                # Step 5: Apply forecast filter
                filtered_weights = {}
                for stock in selected_stocks:
                    if forecasts.get(stock, 0) > 0:  # Only include stocks with positive forecasts
                        filtered_weights[stock] = raw_weights[stock]

                # Renormalize weights
                total_weight = sum(filtered_weights.values())
                if total_weight > 0:
                    final_weights = {stock: w / total_weight for stock, w in filtered_weights.items()}
                else:
                    final_weights = {}

                # Debug final weights
                logger.info(f"   📊 Final weights: {[(k, f'{v:.3f}') for k, v in final_weights.items()]}")
                period_debug['final_weights'] = len(final_weights)
                period_debug['final_weights_sum'] = sum(final_weights.values()) if final_weights else 0

                # Step 6: All-agree signal for ensemble strategies
                if 'allagree' in strategy_name.lower():
                    signals = list(trading_signals.values())
                    if signals:
                        total_signals = sum(signals)
                        all_agree_signal = total_signals / abs(total_signals) if total_signals != 0 else 0
                        logger.info(f"   🤝 All-agree signal: {total_signals} → {all_agree_signal}")
                    else:
                        all_agree_signal = 0

                    # Apply all-agree filter
                    if all_agree_signal <= 0:
                        final_weights = {}  # Go to cash
                        logger.info(f"   💰 Going to cash due to negative all-agree signal")

                    period_debug['all_agree_signal'] = all_agree_signal

                # Step 7: Calculate portfolio return
                portfolio_return = 0.0
                if final_weights:
                    for stock, weight in final_weights.items():
                        if stock in return_data and t+1 < len(return_data[stock]):
                            stock_return = return_data[stock].iloc[t+1]
                            portfolio_return += weight * stock_return
                    successful_trades += 1
                    logger.info(f"   📈 Portfolio return: {portfolio_return:.4f} ({100*portfolio_return:.2f}%)")
                else:
                    logger.info(f"   💰 Holding cash (no positions)")

                period_debug['portfolio_return'] = portfolio_return
                period_debug['status'] = 'success'

                # Update capital
                current_capital *= (1 + portfolio_return)

                # Store results
                portfolio_values.append(current_capital)
                selected_stocks_history.append(selected_stocks.copy())
                weights_history.append(final_weights.copy())
                debug_info.append(period_debug)

            except Exception as e:
                logger.error(f"❌ Error in strategy execution at time {t}: {e}")
                period_debug['status'] = 'error'
                period_debug['error'] = str(e)
                debug_info.append(period_debug)

                # Fallback: maintain current capital
                portfolio_values.append(current_capital)
                selected_stocks_history.append([])
                weights_history.append({})

        # Calculate performance metrics
        total_return = (current_capital - initial_capital) / initial_capital

        # Summary logging
        logger.info(f"🏁 Strategy {strategy_name} complete:")
        logger.info(f"   💰 Total return: {total_return:.4f} ({100*total_return:.2f}%)")
        logger.info(f"   📊 Successful trades: {successful_trades}/{total_periods}")
        logger.info(f"   💸 Final capital: ${current_capital:.2f}")

        return {
            'strategy_name': strategy_name,
            'total_return': total_return,
            'final_capital': current_capital,
            'portfolio_values': portfolio_values,
            'selected_stocks_history': selected_stocks_history,
            'weights_history': weights_history,
            'debug_info': debug_info,  # Add debug info
            'successful_trades': successful_trades,
            'total_periods': total_periods
        }

# =============================================================================
# MAIN EXECUTION ENGINE
# =============================================================================

class MSTPortfolioFramework:
    """Main execution engine for MST-based portfolio strategies"""

    def __init__(self, config: PortfolioConfig):
        self.config = config
        self.data_fetcher = YahooDataFetcher(config)

    def run_complete_analysis(self, start_date: str, end_date: str) -> Dict[str, any]:
        """Run complete MST portfolio analysis"""

        logger.info("Starting MST Portfolio Framework Analysis...")

        # Step 1: Get available stocks
        symbols = self.data_fetcher.get_available_stocks()
        if not symbols:
            raise ValueError("No stock symbols defined")

        logger.info(f"🎯 CONFIRMED: Processing exactly {len(symbols)} symbols")
        logger.info(f"📝 Symbol list: {symbols}")

        if len(symbols) != 35:
            logger.warning(f"⚠️  UNEXPECTED: Got {len(symbols)} symbols instead of 35!")
            symbols = symbols[:35]  # Force limit to 35
            logger.info(f"🔧 FIXED: Truncated to {len(symbols)} symbols")

        # Step 2: Download historical data
        logger.info(f"Downloading historical data for {len(symbols)} symbols...")
        historical_data = self.data_fetcher.get_batch_historical_data(symbols, start_date, end_date)

        if len(historical_data) < 10:
            raise ValueError("Insufficient historical data retrieved")

        # Step 3: Prepare return and price data
        return_data = {}
        price_data = {}

        # Align all data to same date range
        common_dates = None
        for symbol, df in historical_data.items():
            if common_dates is None:
                common_dates = set(df['date'])
            else:
                common_dates = common_dates.intersection(set(df['date']))

        common_dates = sorted(list(common_dates))

        for symbol, df in historical_data.items():
            df_aligned = df[df['date'].isin(common_dates)].sort_values('date').reset_index(drop=True)
            if len(df_aligned) > self.config.window_size + 50:  # Ensure sufficient data
                return_data[symbol] = df_aligned['return']
                price_data[symbol] = df_aligned['adjClose']

        logger.info(f"Prepared aligned data for {len(return_data)} symbols over {len(common_dates)} dates")

        # Step 4: Run multiple strategies
        strategies = [
            'MST + VaR',
            'MST + Sharpe',
            'MST + ARIMA + VaR',
            'MST + ARIMA + Sharpe',
            'MST + NNAR + VaR',
            'MST + NNAR + Sharpe',
            'MST + AllAgree + VaR',
            'MST + AllAgree + Sharpe'
        ]

        results = {}
        strategy_executor = PortfolioStrategy(self.config)

        # Add benchmark (Buy & Hold)
        benchmark_return = self._calculate_benchmark_return(return_data)
        results['Buy & Hold Benchmark'] = {
            'strategy_name': 'Buy & Hold Benchmark',
            'total_return': benchmark_return,
            'final_capital': self.config.initial_capital * (1 + benchmark_return),
            'portfolio_values': [],
            'selected_stocks_history': [],
            'weights_history': []
        }

        # Execute strategies
        for strategy in strategies:
            logger.info(f"Running strategy: {strategy}")
            try:
                result = strategy_executor.run_strategy(strategy, return_data, price_data)
                results[strategy] = result
            except Exception as e:
                logger.error(f"Failed to run strategy {strategy}: {e}")
                continue

        # Step 5: Generate summary report
        summary = self._generate_summary_report(results)

        return {
            'results': results,
            'summary': summary,
            'data_info': {
                'symbols': list(return_data.keys()),
                'date_range': [common_dates[0], common_dates[-1]],
                'n_periods': len(common_dates)
            }
        }

    def _calculate_benchmark_return(self, return_data: Dict[str, pd.Series]) -> float:
        if not return_data:
            raise ValueError("There are no aligned stock returns for the benchmark.")
        total_returns = [(1 + returns).prod() - 1 for returns in return_data.values()]
        return float(np.mean(total_returns))

    def _generate_summary_report(self, results: Dict[str, any]) -> Dict[str, any]:
        """Generate summary report of all strategies"""

        summary = {}
        performance_table = []

        for strategy_name, result in results.items():
            performance_table.append({
                'Strategy': strategy_name,
                'Total Return (%)': f"{result['total_return'] * 100:.2f}",
                'Final Capital': f"${result['final_capital']:.2f}"
            })

        # Sort by total return
        performance_table.sort(key=lambda x: float(x['Total Return (%)'].replace('%', '')), reverse=True)

        summary['performance_table'] = performance_table
        summary['best_strategy'] = performance_table[0]['Strategy'] if performance_table else None
        summary['best_return'] = performance_table[0]['Total Return (%)'] if performance_table else None

        return summary

# =============================================================================
# EXAMPLE USAGE
# =============================================================================

def main():
    """Example usage of the MST Portfolio Framework - Optimized version"""

    # Configuration with optimized parameters for speed
    config = PortfolioConfig(
        window_size=60,   # Reduced for faster processing
        forecast_horizon=5,  # Reduced for speed
        top_k_stocks=5,
        initial_capital=100000.0
    )

    # Initialize framework
    framework = MSTPortfolioFramework(config)

    # Define analysis period - shorter period for faster testing
    end_date = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
    start_date = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')  # 1 year instead of 2

    try:
        # Run complete analysis
        logger.info("="*80)
        logger.info("MST PORTFOLIO FRAMEWORK - OPTIMIZED EXECUTION")
        logger.info("="*80)
        logger.info(f"Analysis period: {start_date} to {end_date}")
        logger.info(f"Configuration: {config.window_size}-day windows, {config.forecast_horizon}-step FEVD")
        logger.info("="*80)

        results = framework.run_complete_analysis(start_date, end_date)

        # Display results
        print("\n" + "="*80)
        print("RESULTS SUMMARY")
        print("="*80)

        for item in results['summary']['performance_table']:
            print(f"{item['Strategy']:<30} {item['Total Return (%)']:>15} {item['Final Capital']:>20}")

        print(f"\n🏆 Best Strategy: {results['summary']['best_strategy']}")
        print(f"📈 Best Return: {results['summary']['best_return']}")

        print(f"\n📊 Analysis Details:")
        print(f"   • Symbols analyzed: {len(results['data_info']['symbols'])}")
        print(f"   • Date range: {results['data_info']['date_range'][0]} to {results['data_info']['date_range'][1]}")
        print(f"   • Trading periods: {results['data_info']['n_periods']}")

        # Performance summary
        n_symbols = len(results['data_info']['symbols'])
        n_pairs = n_symbols * (n_symbols - 1) // 2
        print(f"\n⚡ Performance Summary:")
        print(f"   • FEVD pairs processed: {n_pairs}")
        print(f"   • Execution completed successfully!")

        print("="*80)

    except Exception as e:
        logger.error(f"Analysis failed: {e}")
        raise

if __name__ == "__main__":
    main()
